In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [2]:
from src.paths import DATA_DIR
from src.Qlassifier.baseline import run_tf_idf
from src.Qlassifier.baseline import get_paragraphs, load_data

In [3]:
subjects = ["chemistry", "specialist_mathematics"]
exams = ["2023", "2023_2"]

dfs = []
for subject, exam in zip(subjects, exams):
    subject_path = DATA_DIR / subject
    exam_path = subject_path / "past_exams" / f"{exam}.pdf"
    df = run_tf_idf(exam_path)
    dfs.append(df)

## Evaluation

Since I am no expert in chemistry, I supplied GPT-5 with the necessary documents and asked it to hand-label the best topic classificaitons for us. Unfortunately, for whatever reason, gpt-5 could only reliably label MCQ questions and wasn't very helpful for short answer ones. But alas, even the mcq was of very low quality - by my calculations even worse than this rudimentary TF-IDF approach - as i detected with my limited chemistry knowledge. So, I will hand label on this occasion to the best of my ability, with the help of GPT-5 on individual questions, one at a time.

In [ ]:
true_topic_indices = [ # labelled -1 if i dont know
    0, 2, 6, 5, 10, 9, 0, 4, 0, 4,
    3, 3, 6, 4, 0, 9, 1, 9, 6, 8,
    10 ,2, 6, 2, 5, 11, 12, 2, 9, 2,
]

chem_df = dfs[0]
mcq_chem = chem_df.loc[:29, :].copy()
mcq_chem["True Topic"] = true_topic_indices

mcq_chem.head(5)

,Question,Predicted Topic,Carbon-based fuels (Similarity),Measuring changes in chemical reactions (Similarity),Primary galvanic cells and fuel cells as sources of energy (Similarity),Rates of chemical reactions (Similarity),Extent of chemical reactions (Similarity),Production of chemicals using electrolysis (Similarity),"Structure, nomenclature and properties of organic compounds (Similarity)",Reactions of organic compounds (Similarity),Laboratory analysis of organic compounds (Similarity),Instrumental analysis of organic compounds (Similarity),Medicinal chemistry (Similarity),Investigation design (Similarity),Scientific evidence (Similarity),Science communication (Similarity),Confidence,True Topic
0,Question 1.,0,0.327875,0.058905,0.110483,0.071459,0.091934,0.117756,0.024969,0.069125,0.028245,0.041009,0.040861,0.025480,0.040480,0.019746,1.000000,0
1,Question 2.,2,0.122629,0.107708,0.406177,0.073594,0.096400,0.310753,0.036747,0.070461,0.034924,0.054228,0.076146,0.040570,0.066937,0.030771,0.866113,2
2,Question 3.,6,0.117194,0.110544,0.148962,0.111258,0.172918,0.171263,0.217281,0.128391,0.214496,0.127260,0.103781,0.095613,0.071810,0.067059,0.381076,6
3,Question 4.,5,0.116580,0.087341,0.301472,0.070251,0.134399,0.339391,0.093666,0.087382,0.064139,0.082502,0.108548,0.062268,0.052779,0.043559,0.672543,5
4,Question 5.,10,0.097258,0.120933,0.130148,0.102349,0.144827,0.135652,0.085115,0.136926,0.074359,0.102877,0.171416,0.077593,0.079361,0.062326,0.367179,10


In [6]:
accuracy = sum(mcq_chem["Predicted Topic"] == mcq_chem["True Topic"]) / mcq_chem.shape[0]
print(f"Accuracy on chemistry MCQ {100*accuracy:.1f}%")

Accuracy on chemistry MCQ 73.3%


But our goal isn't really to JUST assign a question to a topic; this is a flawed problem design, since questions may belong to multiple topics. The hand labelling above was just what I deemed to be the single BEST answer. In reality, its the distribution of the weights for each topic that we are interested in. We also didn't test short answer (since the labelling is tedious to say the least, and the one "true label" is even more ambiguous). But this accuracy metric serves as a decent baseline regardless. 

We move on to a mathematics subject, in which we expect our 'model' to perform FAR worse, since we don't have LaTeX data and plenty of the context is in the LaTeX for math exams, especially relative to chemistry ones. This one I hand label reliably.

In [7]:
ground_truth = [
    0, 1, 1, 2, 2, 4, 4, 4, 7, 3, 
    3, 5, 5, 6, 6, 8, 6, 7, 9, 11
]

math_df = dfs[1]
mcq_math = math_df.loc[:19, :].copy()
mcq_math["True Topic"] = ground_truth
mcq_math.head(5)

,Question,Predicted Topic,Logic and proof (Similarity),"Functions, relations and graphs (Similarity)",Complex numbers (Similarity),Differential calculus and integral calculus (Similarity),Differential equations (Similarity),Kinematics: rectilinear motion (Similarity),Vectors (Similarity),Vector and Cartesian equations (Similarity),Vector calculus (Similarity),Distribution of linear combinations of random variables (Similarity),Distribution of the sample mean (Similarity),Confidence intervals for the population mean (Similarity),"Hypothesis testing for a population mean with a sample drawn from a normal distribution of known variance, or for a large sample (Similarity)",Confidence,True Topic
0,Question 1.,0,0.051263,0.012956,0.016780,0.034584,0.026162,0.010739,0.024011,0.013756,0.028125,0.014479,0.046370,0.025569,0.020568,0.540743,0
1,Question 2.,10,0.157929,0.149755,0.137117,0.200027,0.155303,0.153890,0.160399,0.143008,0.154672,0.108404,0.205819,0.145761,0.153603,0.348711,1
2,Question 3.,11,0.061104,0.076285,0.088195,0.085128,0.063305,0.048207,0.094137,0.166653,0.128203,0.056801,0.104982,0.216410,0.088451,0.581226,1
3,Question 4.,9,0.015748,0.000000,0.002394,0.003098,0.002675,0.003196,0.003259,0.001235,0.010314,0.039625,0.027790,0.025192,0.001467,1.000000,2
4,Question 5.,2,0.130946,0.079299,0.175360,0.083131,0.088025,0.098341,0.075158,0.105763,0.097336,0.054169,0.082915,0.077247,0.056693,0.499708,2


In [8]:
accuracy = sum(mcq_math["Predicted Topic"] == mcq_math["True Topic"]) / mcq_math.shape[0]
print(f"Accuracy on specialist math MCQ {100*accuracy:.1f}%")

Accuracy on specialist math MCQ 45.0%


As expected! These results will serve as our baseline. 